<a href="https://colab.research.google.com/github/premt99/Internship_Project/blob/main/Dashboard.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Dashboard of Pollutants

In [17]:
!pip install dash
import pandas as pd
import dash
from dash import dcc, html
import plotly.express as px
from dash.dependencies import Input, Output

# Load dataset
file_path = "/content/24hr Merge Data.csv"  # Update with actual file path
df = pd.read_csv(file_path)

# Convert 'Date' to datetime
df['Date'] = pd.to_datetime(df['Date'], format="%d-%m-%Y", errors='coerce')

# Convert 'Time' to proper format (handling both HH:MM and HH:MM:SS)
df['Datetime'] = pd.to_datetime(df['Date'].astype(str) + ' ' + df['Time'], errors='coerce', format="%Y-%m-%d %H:%M:%S")
df['Datetime'].fillna(pd.to_datetime(df['Date'].astype(str) + ' ' + df['Time'], errors='coerce', format="%Y-%m-%d %H:%M"), inplace=True)

# Use forward fill (ffill) instead of deprecated `fillna(method=...)`
df.ffill(inplace=True)

# Get unique values for dropdowns
locations = df['Location'].dropna().unique()
times = df['Time'].dropna().unique()
pollutants = ['PM2.5', 'PM10', 'NO', 'NO2', 'NOx', 'NH3', 'SO2', 'CO', 'Ozone',
              'Benzene', 'Toluene', 'Eth-Benzene', 'MP-Xylene', 'O-Xylene',
              'RH', 'WS', 'WD', 'SR', 'AT']  # All 19 pollutants

# Initialize Dash app
app = dash.Dash(__name__)

app.layout = html.Div(
    style={'backgroundColor': 'white', 'padding': '20px', 'fontFamily': 'Arial'},
    children=[
        # Header
        html.H1("Air Quality Dashboard", style={'textAlign': 'center', 'color': 'black'}),

        # Dropdowns for selection
        html.Div([
            html.Label("Select Location:", style={'marginRight': '10px'}),
            dcc.Dropdown(
                id="location-dropdown",
                options=[{'label': loc, 'value': loc} for loc in locations],
                value=locations[0] if len(locations) > 0 else None,
                clearable=False,
                style={'width': '200px', 'display': 'inline-block'}
            ),

            html.Label("Select Time:", style={'marginLeft': '20px', 'marginRight': '10px'}),
            dcc.Dropdown(
                id="time-dropdown",
                options=[{'label': time, 'value': time} for time in times],
                value=times[0] if len(times) > 0 else None,
                clearable=False,
                style={'width': '200px', 'display': 'inline-block'}
            ),

            html.Label("Select Pollutant:", style={'marginLeft': '20px', 'marginRight': '10px'}),
            dcc.Dropdown(
                id="pollutant-dropdown",
                options=[{'label': p, 'value': p} for p in pollutants],
                value='PM2.5',
                clearable=False,
                style={'width': '200px', 'display': 'inline-block'}
            ),

            html.Label("View:", style={'marginLeft': '20px', 'marginRight': '10px'}),
            dcc.Dropdown(
                id="view-dropdown",
                options=[
                    {'label': 'Hourly Trend', 'value': 'hourly'},
                    {'label': 'Daily Trend', 'value': 'daily'}
                ],
                value='hourly',
                clearable=False,
                style={'width': '200px', 'display': 'inline-block'}
            )
        ], style={'display': 'flex', 'justifyContent': 'center', 'alignItems': 'center'}),

        # Line chart
        dcc.Graph(id="line-chart", style={'marginTop': '20px'})
    ]
)


# Callback to update the line chart based on selection
@app.callback(
    Output("line-chart", "figure"),
    Input("location-dropdown", "value"),
    Input("time-dropdown", "value"),
    Input("pollutant-dropdown", "value"),
    Input("view-dropdown", "value")
)
def update_chart(selected_location, selected_time, selected_pollutant, selected_view):
    filtered_df = df[df['Location'] == selected_location].copy()

    if selected_view == 'hourly':
        filtered_df = filtered_df[filtered_df['Time'] == selected_time]
        title = f"{selected_pollutant} Levels Over Time in {selected_location} at {selected_time}"
        x_axis = "Datetime"
    else:  # Daily Trend
        filtered_df = filtered_df.groupby('Date')[pollutants].mean().reset_index()
        title = f"Daily Average {selected_pollutant} Levels in {selected_location}"
        x_axis = "Date"

    fig = px.line(
        filtered_df, x=x_axis, y=selected_pollutant, markers=True,
        title=title,
        labels={x_axis: "Date/Time", selected_pollutant: f"{selected_pollutant} Level"}
    )

    # Formatting
    fig.update_layout(
        xaxis_title="Date/Time",
        yaxis_title=f"{selected_pollutant} Level",
        xaxis=dict(showgrid=True),
        yaxis=dict(showgrid=True),
        plot_bgcolor="white",
        margin=dict(l=50, r=50, t=50, b=50)
    )

    return fig


# Run the app
if __name__ == "__main__":
    app.run(debug=True)


<ipython-input-17-cd8acd0b9db0>:17: FutureWarning:

A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.





<IPython.core.display.Javascript object>